In [2]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

In [3]:
with open('NMOS/idW_vs_gmid_vdssweep.vcsv', 'r') as f:
    lines = f.readlines()
   
    for line in lines:
        print(line)
    
    

;Version, 1, 0

;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id" ?result "dc")/VAR("W")  )re/schematic" ?result "dc")) "L" 4.5e-08 "vds" 0.1),;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id" ?result "dc")/VAR("W")  )re/schematic" ?result "dc")) "L" 4.5e-08 "vds" 0.2),;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id" ?result "dc")/VAR("W")  )re/schematic" ?result "dc")) "L" 4.5e-08 "vds" 0.3),;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id" ?result "dc")/VAR("W")  )re/schematic" ?result "dc")) "L" 4.5e-08 "vds" 0.4),;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id" ?result "dc")/VAR("W")  )re/schematic" ?result "dc")) "L" 4.5e-08 "vds" 0.5),;waveVsWave(?x getData("nmos:gm" ?result "dc")/getData("nmos:id" ?result "dc") ?y getData("nmos:id"

In [4]:
header_line = lines[1]
re.findall(r'"L"\s+([0-9\.eE+-]+)\s+"vds"\s+([0-9\.eE+-]+)', header_line)

[('4.5e-08', '0.1'),
 ('4.5e-08', '0.2'),
 ('4.5e-08', '0.3'),
 ('4.5e-08', '0.4'),
 ('4.5e-08', '0.5'),
 ('4.5e-08', '0.6'),
 ('4.5e-08', '0.7'),
 ('4.5e-08', '0.8'),
 ('4.5e-08', '0.9'),
 ('4.5e-08', '1.0'),
 ('4.5e-08', '1.1'),
 ('1.45e-07', '0.1'),
 ('1.45e-07', '0.2'),
 ('1.45e-07', '0.3'),
 ('1.45e-07', '0.4'),
 ('1.45e-07', '0.5'),
 ('1.45e-07', '0.6'),
 ('1.45e-07', '0.7'),
 ('1.45e-07', '0.8'),
 ('1.45e-07', '0.9'),
 ('1.45e-07', '1.0'),
 ('1.45e-07', '1.1'),
 ('2.45e-07', '0.1'),
 ('2.45e-07', '0.2'),
 ('2.45e-07', '0.3'),
 ('2.45e-07', '0.4'),
 ('2.45e-07', '0.5'),
 ('2.45e-07', '0.6'),
 ('2.45e-07', '0.7'),
 ('2.45e-07', '0.8'),
 ('2.45e-07', '0.9'),
 ('2.45e-07', '1.0'),
 ('2.45e-07', '1.1'),
 ('3.45e-07', '0.1'),
 ('3.45e-07', '0.2'),
 ('3.45e-07', '0.3'),
 ('3.45e-07', '0.4'),
 ('3.45e-07', '0.5'),
 ('3.45e-07', '0.6'),
 ('3.45e-07', '0.7'),
 ('3.45e-07', '0.8'),
 ('3.45e-07', '0.9'),
 ('3.45e-07', '1.0'),
 ('3.45e-07', '1.1'),
 ('4.45e-07', '0.1'),
 ('4.45e-07', '0.2'),

In [5]:
df_raw = pd.read_csv('NMOS/idW_vs_gmid_vdssweep.vcsv', skiprows=6, header=None)
print(df_raw)

          0           1          2           3          4           5    \
0   29.605202    0.000003  29.304003    0.000005  28.915834    0.000006   
1   29.701153    0.000006  29.561978    0.000009  29.371632    0.000012   
2   29.723642    0.000012  29.659723    0.000017  29.567139    0.000023   
3   29.715267    0.000024  29.684425    0.000033  29.638204    0.000043   
4   29.690428    0.000046  29.672534    0.000064  29.647086    0.000083   
5   29.651716    0.000088  29.636946    0.000123  29.619486    0.000159   
6   29.555186    0.000169  29.538089    0.000236  29.521893    0.000305   
7   29.429618    0.000323  29.406747    0.000451  29.387721    0.000583   
8   29.272935    0.000617  29.241328    0.000860  29.216305    0.001112   
9   29.073395    0.001172  29.029620    0.001633  28.995466    0.002109   
10  28.815619    0.002216  28.755370    0.003083  28.708506    0.003980   
11  28.480157    0.004163  28.398115    0.005782  28.334318    0.007454   
12  28.043707    0.007754

In [ ]:
frames = []
for i, (l_val, vds_val) in enumerate(re.findall(r'"L"\s+([0-9\.eE+-]+)\s+"vds"\s+([0-9\.eE+-]+)', lines[1])):
    x_col = i * 2
    y_col = i * 2 + 1
        
    # Isolate the X (gm/Id) and Y data for this specific L and Vds
    temp_df = df_raw.iloc[:, [x_col, y_col]].copy()
    temp_df.columns = ['gm_Id', 'idW']
    
    # Append the physical parameters
    temp_df['L'] = float(l_val)
    temp_df['VDS'] = float(vds_val)
        
    # Add an index to ensure perfect row-matching when we merge datasets
    temp_df['sweep_index'] = temp_df.index 
        
    frames.append(temp_df)
print(frames)

            0           1
0   29.605202    0.000003
1   29.701153    0.000006
2   29.723642    0.000012
3   29.715267    0.000024
4   29.690428    0.000046
5   29.651716    0.000088
6   29.555186    0.000169
7   29.429618    0.000323
8   29.272935    0.000617
9   29.073395    0.001172
10  28.815619    0.002216
11  28.480157    0.004163
12  28.043707    0.007754
13  27.480617    0.014285
14  26.766464    0.025952
15  25.884280    0.046327
16  24.832955    0.080956
17  23.635344    0.138004
18  22.341606    0.228866
19  21.023418    0.368748
20  19.758675    0.577400
21  18.612126    0.880361
22  17.620120    1.311020
23  16.784818    1.913531
24  16.077761    2.746128
25  15.449332    3.883860
26  14.840541    5.419188
27  14.194751    7.458324
28  13.467868   10.111251
29  12.636115   13.474624
30  11.701057   17.610021
31  10.690556   22.524821
32   9.652059   28.165534
33   8.637495   34.428350
34   7.687911   41.180977
35   6.826806   48.284696
36   6.062145   55.609798
37   5.39169

In [9]:
def parse_cadence_vcsv(filepath, y_col_name):
    """Parses a Cadence waveVsWave CSV and converts it to a long-format DataFrame."""
    with open(filepath, 'r') as f:
        lines = f.readlines() # Read the file line by line
        
    # Extract L and VDS values from the second line using Regex
    header_line = lines[1]
    params = re.findall(r'"L"\s+([0-9\.eE+-]+)\s+"vds"\s+([0-9\.eE+-]+)', header_line)
    
    # Load the numerical data, skipping the 6 Cadence header rows as they are not needed
    df_raw = pd.read_csv(filepath, skiprows=6, header=None)
    
    frames = []
    for i, (l_val, vds_val) in enumerate(params):
        x_col = i * 2
        y_col = i * 2 + 1
        
        # Isolate the X (gm/Id) and Y data for this specific L and Vds
        temp_df = df_raw.iloc[:, [x_col, y_col]].copy()
        temp_df.columns = ['gm_Id', y_col_name]
        
        # Append the physical parameters
        temp_df['L'] = float(l_val)
        temp_df['VDS'] = float(vds_val)
        
        # Add an index to ensure perfect row-matching when we merge datasets
        temp_df['sweep_index'] = temp_df.index 
        
        frames.append(temp_df)
        
    return pd.concat(frames, ignore_index=True).dropna()

df_idw = parse_cadence_vcsv('NMOS/idW_vs_gmid_vdssweep.vcsv', 'Id_W')
print(df_idw)
df_gain = parse_cadence_vcsv('NMOS/gmgds_vs_gmid_vdssweep.vcsv', 'gm_gds')
print(df_gain)

          gm_Id        Id_W             L  VDS  sweep_index
0     29.605202    0.000003  4.500000e-08  0.1            0
1     29.701153    0.000006  4.500000e-08  0.1            1
2     29.723642    0.000012  4.500000e-08  0.1            2
3     29.715267    0.000024  4.500000e-08  0.1            3
4     29.690428    0.000046  4.500000e-08  0.1            4
...         ...         ...           ...  ...          ...
2800   2.835899  122.909099  4.450000e-07  1.1           46
2801   2.714010  130.635627  4.450000e-07  1.1           47
2802   2.599835  138.488163  4.450000e-07  1.1           48
2803   2.492676  146.456075  4.450000e-07  1.1           49
2804   2.391915  154.528953  4.450000e-07  1.1           50

[2805 rows x 5 columns]
          gm_Id     gm_gds             L  VDS  sweep_index
0     29.605202   7.533102  4.500000e-08  0.1            0
1     29.701153   7.531758  4.500000e-08  0.1            1
2     29.723642   7.529842  4.500000e-08  0.1            2
3     29.715267   7

In [10]:
df_nmos = pd.merge(df_idw, df_gain, on=['L', 'VDS', 'sweep_index'])
df_nmos = df_nmos.rename(columns={'gm_Id_x': 'gm_Id'}).drop(columns=['sweep_index', 'gm_Id_y'])
df_nmos = df_nmos[(df_nmos['gm_Id'] >= 2.0) & (df_nmos['gm_Id'] <= 25.0)]
print(df_nmos)

          gm_Id        Id_W             L  VDS     gm_gds
16    24.832955    0.080956  4.500000e-08  0.1   6.733812
17    23.635344    0.138004  4.500000e-08  0.1   6.531060
18    22.341606    0.228866  4.500000e-08  0.1   6.303466
19    21.023418    0.368748  4.500000e-08  0.1   6.060994
20    19.758675    0.577400  4.500000e-08  0.1   5.816351
...         ...         ...           ...  ...        ...
2800   2.835899  122.909099  4.450000e-07  1.1  26.529693
2801   2.714010  130.635627  4.450000e-07  1.1  25.318414
2802   2.599835  138.488163  4.450000e-07  1.1  24.146209
2803   2.492676  146.456075  4.450000e-07  1.1  23.011069
2804   2.391915  154.528953  4.450000e-07  1.1  21.911011

[1858 rows x 5 columns]


In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

print("X transformed")
print(scaler_X.fit_transform(X))

print("Y transformed")
print(scaler_y.fit_transform(y))

[[ 2.26855129 -1.39894102 -1.69360982]
 [ 2.09180628 -1.39894102 -1.69360982]
 [ 1.90087469 -1.39894102 -1.69360982]
 ...
 [-1.01264178  1.41636622  1.52521899]
 [-1.02845638  1.41636622  1.52521899]
 [-1.04332683  1.41636622  1.52521899]]
[[-0.81990574 -0.89340461]
 [-0.81920503 -0.90541409]
 [-0.81808897 -0.918895  ]
 ...
 [ 0.88014474  0.1379731 ]
 [ 0.9780143   0.07073608]
 [ 1.07717316  0.00557699]]


In [13]:
X = df_nmos[['gm_Id', 'L', 'VDS']].values
y = df_nmos[['Id_W', 'gm_gds']].values 
print(X.shape)
print(X)

print(y.shape)
print(y)

(1858, 3)
[[2.48329553e+01 4.50000000e-08 1.00000000e-01]
 [2.36353442e+01 4.50000000e-08 1.00000000e-01]
 [2.23416060e+01 4.50000000e-08 1.00000000e-01]
 ...
 [2.59983484e+00 4.45000000e-07 1.10000000e+00]
 [2.49267627e+00 4.45000000e-07 1.10000000e+00]
 [2.39191522e+00 4.45000000e-07 1.10000000e+00]]
(1858, 2)
[[8.09560131e-02 6.73381178e+00]
 [1.38004039e-01 6.53105991e+00]
 [2.28866139e-01 6.30346638e+00]
 ...
 [1.38488163e+02 2.41462088e+01]
 [1.46456075e+02 2.30110691e+01]
 [1.54528953e+02 2.19110105e+01]]
